# End-to-End Classification (NLP) Pipeline — SMS Spam (Assignment)

## Goal
Take **raw SMS text** and build a deployable spam classifier. You'll write the full NLP preprocessing chain, wrap it inside a sklearn `Pipeline`, tune it, tune the decision threshold for imbalance, and save the artifact.

Each step tells you *what* to implement and *why*. Write the code yourself in the empty cells. When stuck, check `02_classification_nlp_end_to_end.ipynb` for the worked solution.

## Dataset
SMS Spam Collection — 5,574 SMS messages labeled `ham` or `spam`, ~13% spam (imbalanced). The TSV file is bundled in this repo at `module4/4.3_classification_models/SMSSpamCollection`.

## Deliverable
- `artifacts_classification/spam_model.pkl` and `.joblib`
- `artifacts_classification/metadata.json` — versions, metrics, best threshold
- A working reload + raw-text smoke test

---

## Step 0: Imports

All libraries you'll need are pre-loaded below — just run the cell.

In [ ]:
import os, re, json, pickle, joblib, sys
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score
)
import sklearn
import warnings
warnings.filterwarnings('ignore')

# Optional NLP toolkit — used in Step 4 if available
try:
    import nltk
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('wordnet', quiet=True)
    from nltk.corpus import stopwords as nltk_stop
    from nltk.stem import PorterStemmer, WordNetLemmatizer
    HAS_NLTK = True
except Exception:
    HAS_NLTK = False

plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42
print('Imports OK  |  NLTK available:', HAS_NLTK)

## Step 1: Define the problem

**Task:** in markdown, answer:
1. Goal — what classes?
2. Cost asymmetry — which mistake is worse, false positive (ham flagged as spam) or false negative (spam slips through)?
3. Which metric matters most given that asymmetry?
4. Acceptance criterion (e.g. F1 > 0.9 on spam class)?

**Why:** the cost structure determines which threshold and metrics you optimize.

**YOUR ANSWERS**
1. 
2. 
3. 
4. 

## Step 2: Load the data

Dataset is pre-loaded below. Just run the cell.

In [ ]:
PATH = '../module4/4.3_classification_models/SMSSpamCollection'
if not os.path.exists(PATH):
    PATH = '/Users/kmantha/ML_LEARNING_TRAINING/ml_modules/module4/4.3_classification_models/SMSSpamCollection'

df = pd.read_csv(PATH, sep='\t', header=None, names=['label', 'text'])
df['target'] = (df['label'] == 'spam').astype(int)
print(f'Shape: {df.shape}')
df.head()

## Step 3: EDA — what does the text look like?

**Task:**
1. Print class balance (`value_counts(normalize=True)`).
2. Add columns `len_chars` and `len_words`. Plot histograms of message length per class (overlapping, with alpha).
3. Print 3 sample ham messages and 3 sample spam messages — read them.

**For text data,** lengths and sample messages tell you *what tokens you'll need* in preprocessing (URLs? phone numbers? all-caps?).

In [ ]:
# YOUR CODE HERE


## Step 4: NLP preprocessing function

**Task:** write `clean_text(t, keep_negation=False, do_stem=True)` that does:
1. Lowercase.
2. Replace URLs with `urltoken`, long digit runs with `phonetoken`, other digits with `numtoken` — keeps signal, drops literal value.
3. Strip HTML and any non-letter/non-apostrophe characters.
4. Collapse whitespace and tokenize on whitespace.
5. Remove stopwords (use `nltk.corpus.stopwords` if available, else a small built-in set). If `keep_negation=True`, KEEP `not`, `no`, `never`.
6. If `do_stem=True`, apply Porter stemming.
7. Return the cleaned space-joined string.

Demo it on 3 messages — print original and cleaned side by side.

**Why placeholder tokens?** A literal phone number `08452810075` is noise. The *fact that there is a phone number* is signal. Keep the signal, drop the noise.

In [ ]:
# YOUR CODE HERE


## Step 5: Wrap the cleaner as a sklearn Transformer

**Task:** create a class `TextCleaner(BaseEstimator, TransformerMixin)` with `__init__`, `fit` (no-op), and `transform` (apply `clean_text` to each item).

**Why:** so it slots into a `Pipeline` and runs on the right fold inside cross-validation — no "forgot to clean the test set" bugs.

In [ ]:
# YOUR CODE HERE


## Step 6: Stratified train/test split

**Task:** split `X = df['text']`, `y = df['target']` 80/20, `stratify=y`, `random_state=RANDOM_STATE`. Print the spam fraction in train and test — they should match.

**Why stratify?** Without it, a random split could leave too few spam messages in the test set to evaluate properly.

In [ ]:
# YOUR CODE HERE


## Step 7: Build the full Pipeline

**Task:** write a function `make_pipe(model)` that returns:
`TextCleaner → TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95) → model`

**Why include TextCleaner inside Pipeline?** At inference time, your API gets raw SMS text. The Pipeline applies cleaning + vectorization automatically. Don't make the caller do it.

In [ ]:
# YOUR CODE HERE


## Step 8: Baseline — predict majority class

**Task:** fit `DummyClassifier(strategy='most_frequent')`. Print accuracy AND F1 on the spam class.

**Why:** to demonstrate that ~87% accuracy is meaningless on imbalanced data — F1 on the minority class is what counts.

In [ ]:
# YOUR CODE HERE


## Step 9: Bake-off — try several models

**Task:** score 4 candidates with 5-fold StratifiedKFold CV using `scoring='f1'`:
- `MultinomialNB`
- `LogisticRegression(max_iter=1000, class_weight='balanced')`
- `LinearSVC(class_weight='balanced')`
- `RandomForestClassifier(n_estimators=200, class_weight='balanced')`

Wrap each with `make_pipe(...)`. Print mean ± std for each.

In [ ]:
# YOUR CODE HERE


## Step 10: Tune the winner with GridSearchCV

**Task:** for the best CV model (likely Logistic Regression), grid-search:
- `tfidf__ngram_range` ∈ {(1,1), (1,2)}
- `tfidf__min_df` ∈ {1, 2}
- `tfidf__max_df` ∈ {0.9, 0.95}
- `model__C` ∈ {0.5, 1.0, 5.0}

Use `cv=3`, `scoring='f1'`. Print best params and best CV F1.

**Tip:** the `step_name__param` syntax lets GridSearch reach into the Pipeline.

In [ ]:
# YOUR CODE HERE


## Step 11: Threshold tuning for imbalance

**Task:**
1. Get `y_prob = grid.best_estimator_.predict_proba(X_test)[:, 1]`.
2. Compute `precision_recall_curve` and F1 at every threshold.
3. Find the threshold that maximizes F1.
4. Plot the PR curve (mark best point in red) and F1-vs-threshold next to it.
5. Print F1 at default 0.5 vs F1 at the best threshold.

**Why:** `predict()` uses 0.5 by default. With imbalance a different cutoff often gives substantially better precision/recall trade-off.

In [ ]:
# YOUR CODE HERE


## Step 12: Final evaluation

**Task:** apply the best threshold to get `y_pred`. Print:
- `classification_report` with `target_names=['ham','spam']`
- ROC-AUC

Plot side-by-side:
- Confusion matrix as a heatmap.
- ROC curve.

In [ ]:
# YOUR CODE HERE


## Step 13: Inspect mistakes

**Task:** build a DataFrame with columns `text, actual, pred, prob`. Print 5 false positives (sorted by highest prob) and 5 false negatives (sorted by lowest prob).

**Why:** numbers don't tell you *what kind* of mistakes the model makes. Reading 10 wrong examples reveals patterns that drive the next iteration.

In [ ]:
# YOUR CODE HERE


## Step 14: Explain the model — top features per class

**Task:** pull `vocab` from the TF-IDF step and `coefs` from the LR step (use `.named_steps`). Print the top 15 most spam-y tokens (largest positive coefs) and the top 15 most ham-y tokens (largest negative coefs).

In [ ]:
# YOUR CODE HERE


## Step 15: Save the artifact

**Task:**
1. Make folder `artifacts_classification/`.
2. Save the full pipeline with both pickle and joblib.
3. Save metadata to `metadata.json` including: model name/version/timestamp, **best_threshold**, metrics (roc_auc, f1, precision/recall on spam), best params, env versions, preprocessing notes.

**Why include the threshold?** It's part of the model's behavior — without it, callers default to 0.5 and your tuning is wasted.

In [ ]:
# YOUR CODE HERE


## Step 16: Reload + smoke test on raw text

**Task:** load `spam_model.joblib`. Read the threshold from `metadata.json`. Predict on these strings:
- 'Hey, are we still on for lunch tomorrow?'
- 'CONGRATS! You won a free iPhone. Click http://bit.ly/xxx to claim now!'
- 'Call me when you finish work'
- 'URGENT! Your account has been suspended. Reply YES to reactivate. Charges apply.'

Print each as `[SPAM] (p=0.97) text...` or `[HAM]`.

**Why pass raw text?** Confirms the entire preprocessing → vectorizer → model chain works end-to-end as a single artifact.

In [ ]:
# YOUR CODE HERE


---
## Self-check questions

1. Why did you put `TextCleaner` *inside* the Pipeline rather than calling it on `X_train` once before fitting?
2. The dataset is ~87% ham / 13% spam. What metric should you NOT report as your main number, and why?
3. You see `urltoken` in the top spam features. Is that a bug or a feature? Explain.
4. Your best threshold is 0.32, not 0.5. What does that say about the cost of false positives vs false negatives in this run?
5. A new model regressed in production. Walk through 3 things you'd check first using your saved metadata.

**YOUR ANSWERS**
1. 
2. 
3. 
4. 
5. 